# P1.01 · Decision API (FastAPI)
**End-to-end credit decision engine** · **L3** tier

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1thirteeng3/ds-ai-cookbook/blob/colab/en/projects/P1-credit/P1.01-e2e-credit-decision-engine/notebooks/33-serving-api.ipynb)

| Level | Colab runtime | Time (smoke / full) | Document |
|---|---|---|---|
| 🟠 L3 | cpu | 2 / 5 min | [index.en.md](../index.en.md) |

> Packages model, calibrator, vocabulary and policy into a FastAPI service that takes the raw application, returns PD, decision and reasons, and proves it matches training.

## 0. Parameters

`RUN_MODE="smoke"` uses 10% of the data (or the synthetic twin with `OFFLINE=True`) and runs in minutes. `RUN_MODE="full"` uses the whole dataset and produces the official results. `OFFLINE=True` exists only for tests and CI: never draw conclusions from it.

In [ ]:
# Parameters (papermill overrides this cell)
RUN_MODE = "smoke"   # "smoke" | "full"
SEED = 42
DATA_DIR = "data"    # Colab + Drive: "/content/drive/MyDrive/cookbook-data"
OFFLINE = False      # True: synthetic twin (tests/CI). Never for full results.
SAMPLE_FRAC = 0.1    # smoke only
N_BOOT = 200         # bootstrap resamples for metric uncertainty
N_REQUESTS = 1000   # latency test size in full mode (smoke uses 100)

## 1. Setup

On Colab, installs the `cookbook` package from the private repo (`GH_TOKEN` secret) and the exact versions used for the results. NumPy, pandas and scikit-learn keep Colab's versions and are recorded in the result file. If the install upgrades any of them, use *Runtime > Restart session* and run from section 0. To download Home Credit, create the `KAGGLE_API_TOKEN` secret (or `KAGGLE_USERNAME` and `KAGGLE_KEY`) and accept the competition rules on kaggle.com with the same account.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = {
    "optbinning": "1.0.0",
    "lightgbm": "4.7.0",
    "optuna": "5.0.0",
    "mlflow": "3.18.0",
    "fastapi": "0.162.0",
    "httpx": "0.28.1",
    "polars": "2.1.0",
    "pyarrow": "26.0.1",
    "kaggle": "2.2.5",
    "joblib": "1.6.0",
    "matplotlib": "3.11.2",
}

if IN_COLAB and not sys.modules.get("cookbook"):
    from google.colab import userdata

    token = userdata.get("GH_TOKEN")
    pins = " ".join(f"{k}=={v}" for k, v in PACKAGES.items())
    %pip install -q uv
    !uv pip install --system -q {pins} "cookbook[credit,kaggle] @ git+https://{token}@github.com/1thirteeng3/cookbokk.git@main"

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from cookbook import credit as cr
from cookbook import home_credit as hc
from cookbook.data import sample
from cookbook.env import colab_secrets_to_env, hardware_info, quiet_logs, set_seeds, versions
from cookbook.results import Timer, save_result

quiet_logs()  # before heavy imports: silences optional-solver probing
import importlib.util
import json
import time

set_seeds(SEED)
colab_secrets_to_env()
if RUN_MODE == "full" and OFFLINE:
    raise ValueError("RUN_MODE='full' needs real data (OFFLINE=False).")

SOURCE = "home-credit-synthetic" if OFFLINE else "home-credit"
ARTIFACTS = Path(DATA_DIR) / "artifacts" / "P1.01" / RUN_MODE
ARTIFACTS.mkdir(parents=True, exist_ok=True)
HW = hardware_info()
VERSIONS = versions([*PACKAGES, "numpy", "pandas", "scikit-learn"])
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 140)
print(SOURCE, RUN_MODE, ARTIFACTS, HW)

In [ ]:
missing = [f for f in ["lgbm.txt", "calibrator.joblib", "feature_meta.json", "policy.json", "model_card.json", "l3_predictions.parquet"] if not (ARTIFACTS / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {ARTIFACTS}. Run notebooks 30, 31 and 32 first "
        f"with the same RUN_MODE and DATA_DIR. / Rode 30, 31 e 32 antes."
    )

## 2. Service design

- **Input:** the raw application (`application_train` columns) and the list of bureau credits, as an origination system would send them.
- **Processing:** the **same** `hc.build_features` function used in training, the frozen vocabulary, the model, the calibrator and the cut-off from `policy.json`.
- **Output:** PD, points score, decision, three reason codes and the model version.

The service code is written to `serving/app.py` together with a `Dockerfile`. Below it is imported from the file and tested with `TestClient`, without starting a server.

In [ ]:
SERVICE_CODE = """
\"\"\"P1.01 credit decision API. Generated by notebook 33; run: uvicorn app:app --port 8080\"\"\"

import json
import os
from pathlib import Path

import joblib
import lightgbm as lgb
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from cookbook import credit as cr
from cookbook import home_credit as hc

MODEL_DIR = Path(os.environ.get("MODEL_DIR", Path(__file__).resolve().parents[1]))


class BureauRecord(BaseModel):
    CREDIT_ACTIVE: str
    DAYS_CREDIT: int
    CREDIT_DAY_OVERDUE: int = 0
    AMT_CREDIT_SUM: float | None = None
    AMT_CREDIT_SUM_DEBT: float | None = None
    AMT_CREDIT_SUM_OVERDUE: float | None = None
    AMT_CREDIT_MAX_OVERDUE: float | None = None
    CREDIT_TYPE: str = "Consumer credit"


class Application(BaseModel):
    SK_ID_CURR: int
    fields: dict[str, float | int | str | None] = Field(description="application_train columns")
    bureau: list[BureauRecord] = []


class Decision(BaseModel):
    SK_ID_CURR: int
    pd: float
    score: float
    decision: str
    reasons: list[str]
    model_version: str


def create_app(model_dir: Path = MODEL_DIR) -> FastAPI:
    booster = lgb.Booster(model_file=str(model_dir / "lgbm.txt"))
    calibrator = joblib.load(model_dir / "calibrator.joblib")
    meta = json.loads((model_dir / "feature_meta.json").read_text())
    policy = json.loads((model_dir / "policy.json").read_text())
    card = json.loads((model_dir / "model_card.json").read_text())
    cols, version = meta["model_columns"], f"{card['model_name']}:{card['model_version']}"
    known = set(hc.APPLICATION_COLUMNS) - {hc.TARGET}
    api = FastAPI(title="P1.01 credit decision API", version=version)

    @api.get("/health")
    def health() -> dict:
        return {"status": "ok", "model_version": version, "threshold_pd": policy["threshold_pd"]}

    @api.post("/score", response_model=Decision)
    def score(req: Application) -> Decision:
        unknown = set(req.fields) - known
        if unknown:
            raise HTTPException(422, f"unknown fields: {sorted(unknown)}")
        row = dict.fromkeys(known) | req.fields | {hc.ID: req.SK_ID_CURR, hc.TARGET: 0}
        bureau = pd.DataFrame([b.model_dump() | {hc.ID: req.SK_ID_CURR} for b in req.bureau],
                              columns=hc.BUREAU_COLUMNS)
        feats = hc.build_features(pd.DataFrame([row]), bureau)
        X, _ = hc.to_categorical(feats[cols], meta["categories"])
        raw = booster.predict(X)
        pd_ = float(calibrator.predict(raw)[0])
        contrib = booster.predict(X, pred_contrib=True)[:, :-1]
        return Decision(
            SK_ID_CURR=req.SK_ID_CURR, pd=pd_, score=float(cr.score_from_pd(pd_)),
            decision="approve" if pd_ <= policy["threshold_pd"] else "decline",
            reasons=cr.reason_codes(contrib, cols, k=3)[0], model_version=version,
        )

    return api


app = create_app() if (MODEL_DIR / "lgbm.txt").exists() else None
"""

DOCKERFILE = """
FROM python:3.12-slim
WORKDIR /srv
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . /srv/model
ENV MODEL_DIR=/srv/model
WORKDIR /srv/model/serving
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8080"]
"""

serving = ARTIFACTS / "serving"
serving.mkdir(exist_ok=True)
(serving / "app.py").write_text(SERVICE_CODE.strip() + "\n")
(serving / "Dockerfile").write_text(DOCKERFILE.strip() + "\n")
(ARTIFACTS / "requirements.txt").write_text(
    "\n".join(f"{k}=={v}" for k, v in PACKAGES.items() if k in {"lightgbm", "fastapi", "polars", "pyarrow", "joblib"})
    + "\nuvicorn\ncookbook  # this repository's package: pip install from the repo\n"
)
sorted(p.name for p in serving.iterdir())

In [ ]:
spec = importlib.util.spec_from_file_location("p101_app", ARTIFACTS / "serving" / "app.py")
service = importlib.util.module_from_spec(spec)
spec.loader.exec_module(service)
api = service.create_app(ARTIFACTS)

from fastapi.testclient import TestClient
client = TestClient(api)
client.get("/health").json()

## 3. Requests from the raw application

We reload the raw data and build requests for **test** clients exactly as an external system would send them: missing values as `null`, bureau as a list.

In [ ]:
n_twin = int(307_511 * (SAMPLE_FRAC if RUN_MODE == "smoke" else 1))
app, bureau, DATASET = hc.load(SOURCE, DATA_DIR, n_synthetic=n_twin, seed=SEED)
if RUN_MODE == "smoke" and not OFFLINE:
    app = sample(app, SAMPLE_FRAC, SEED, stratify=hc.TARGET)
    bureau = bureau[bureau[hc.ID].isin(app[hc.ID])]
DATASET["sample_frac"] = SAMPLE_FRAC if RUN_MODE == "smoke" else 1.0
DATASET["n_rows"] = len(app)
print(app.shape, bureau.shape, f"default rate = {app[hc.TARGET].mean():.4f}")

In [ ]:
l3 = pd.read_parquet(ARTIFACTS / "l3_predictions.parquet")
test_ids = l3.loc[l3["SPLIT"] == "test", hc.ID]
n_req = N_REQUESTS if RUN_MODE == "full" else 100
ids = test_ids.sample(min(n_req, len(test_ids)), random_state=SEED).to_numpy()
raw = app.set_index(hc.ID)
bureau_by_id = {k: g for k, g in bureau.groupby(hc.ID)}

def to_json_value(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    return v.item() if hasattr(v, "item") else v

def make_request(i):
    fields = {c: to_json_value(v) for c, v in raw.loc[i].drop(labels=[hc.TARGET]).items()}
    hist = bureau_by_id.get(i, pd.DataFrame(columns=hc.BUREAU_COLUMNS)).drop(columns=hc.ID)
    records = [{k: to_json_value(v) for k, v in r.items()} for r in hist.to_dict("records")]
    return {"SK_ID_CURR": int(i), "fields": fields, "bureau": records}

requests = [make_request(i) for i in ids]
example = client.post("/score", json=requests[0]).json()
example

## 4. Validation

### 4.1 Training/serving parity

The PD returned by the API must be **identical** to the one computed in training for the same client. A difference above 1e-9 means the two computation paths diverge.

### 4.2 Latency

Time per request on `TestClient` (in process, no network). Useful to compare versions, not as a production number.

In [ ]:
timings, api_pd = [], []
for body in requests:
    t0 = time.perf_counter()
    r = client.post("/score", json=body)
    timings.append((time.perf_counter() - t0) * 1000)
    assert r.status_code == 200, r.text
    api_pd.append(r.json()["pd"])

batch_pd = l3.set_index(hc.ID).loc[ids, "pd_l3"].to_numpy()
PARITY = float(np.max(np.abs(np.array(api_pd) - batch_pd)))
LAT = {"p50": float(np.percentile(timings, 50)), "p95": float(np.percentile(timings, 95))}
assert PARITY < 1e-9, f"training/serving skew: max |diff| = {PARITY}"
print(f"parity max |diff| = {PARITY:.2e} · latency p50 = {LAT['p50']:.1f} ms · p95 = {LAT['p95']:.1f} ms")

## 5. Variation axes

### 5.1 Bureau history size

Bureau aggregation runs on every request. Does latency grow with the number of previous credits?

In [ ]:
n_bureau = np.array([len(b["bureau"]) for b in requests])
bands = pd.cut(n_bureau, [-1, 0, 3, 8, 1000], labels=["0", "1-3", "4-8", "9+"])
pd.DataFrame({"credits": bands, "ms": timings}).groupby("credits", observed=True)["ms"].describe()[["count", "50%", "max"]]

## 6. Pitfalls demonstrated

### 6.1 Rebuilding features by hand in the API

**Symptom:** API PDs differ from training PDs, with no error at all. **Cause:** an "equivalent" reimplementation that forgets a detail (here, the 365243 sentinel handling). **Fix:** import the same function and test parity in CI.

### 6.2 Unknown category

An occupation that did not exist in training must not break the API or shift codes: the frozen vocabulary turns it into missing.

In [ ]:
meta = json.loads((ARTIFACTS / "feature_meta.json").read_text())
booster = service.lgb.Booster(model_file=str(ARTIFACTS / "lgbm.txt"))
calibrator = service.joblib.load(ARTIFACTS / "calibrator.joblib")

def handmade_features(app_rows, bureau_rows):
    f = hc.engineer(app_rows)                      # forgot hc.clean(): sentinel stays a number
    f["DAYS_EMPLOYED_ANOM"] = 0
    f = f.merge(hc.aggregate_bureau(bureau_rows), on=hc.ID, how="left")
    f["BUREAU_HAS_HISTORY"] = f["BUREAU_COUNT"].notna().astype(int)
    return f

pens = app[app["DAYS_EMPLOYED"] == hc.DAYS_EMPLOYED_ANOMALY].head(50)
sub_bureau = bureau[bureau[hc.ID].isin(pens[hc.ID])]
right = hc.build_features(pens, sub_bureau)
wrong = handmade_features(pens, sub_bureau)
pd_right = calibrator.predict(booster.predict(hc.to_categorical(right[meta["model_columns"]], meta["categories"])[0]))
pd_wrong = calibrator.predict(booster.predict(hc.to_categorical(wrong[meta["model_columns"]], meta["categories"])[0]))
print(f"pensioners: max |PD diff| with hand-made features = {np.max(np.abs(pd_right - pd_wrong)):.4f}")

odd = requests[0] | {"fields": requests[0]["fields"] | {"OCCUPATION_TYPE": "Astronaut"}}
client.post("/score", json=odd).json()

## 7. Results

In [ ]:
DATASET["n_rows"] = len(requests)
save_result(
    recipe_id="P1.01", notebook="33-serving-api", stack="fastapi-testclient", run_mode=RUN_MODE,
    hardware=HW, versions=VERSIONS, dataset=DATASET, params={"n_requests": len(requests)},
    metrics={
        "latency_p50_ms": {"mean": round(LAT["p50"], 3), "std": None, "n": len(requests)},
        "latency_p95_ms": {"mean": round(LAT["p95"], 3), "std": None, "n": len(requests)},
        "parity_max_abs_diff": {"mean": PARITY, "std": None, "n": len(requests)},
    },
    time_s={"train": 0.0, "inference_per_1k": round(float(np.mean(timings)), 6)},
)

## 8. Conclusion

- A service that reproduces training bit for bit, explains every decision and returns the model version.
- For production: run the `Dockerfile` image behind a gateway and move the parity test into CI (recipes G1.02, G1.04 and G2.04).
- **Next step:** [34 · monitoring](34-monitoring.ipynb).